## Without web search

In [2]:
from dotenv import load_dotenv
import os
from pydantic import BaseModel

load_dotenv()
model=os.getenv("OPENAI_MODEL_NAME")

In [18]:
class ResponseFormat(BaseModel):
    Input_Query:str
    Response:str

In [19]:
from langchain.agents import create_agent

agent = create_agent(
    model=model,
    response_format=ResponseFormat
)

In [20]:
from langchain.messages import HumanMessage

question = HumanMessage(content="How up to date is your training knowledge?")

response = agent.invoke(
    {"messages": [question]}
)

In [21]:
print(response['messages'][-1].content)

{"Input_Query":"How up to date is your training knowledge?","Response":"My training knowledge is up to date until October 2023."}


## Add web search tool

In [22]:
from langchain.tools import tool
from typing import Dict, Any
from tavily import TavilyClient

tavily_client = TavilyClient()

@tool
def web_search(query: str) -> Dict[str, Any]:

    """Search the web for information"""

    return tavily_client.search(query)

web_search.invoke("Who is the current mayor of San Francisco?")

{'query': 'Who is the current mayor of San Francisco?',
 'follow_up_questions': None,
 'answer': None,
 'images': [],
 'results': [{'url': 'https://en.wikipedia.org/wiki/Mayor_of_San_Francisco',
   'title': 'Mayor of San Francisco - Wikipedia',
   'content': 'There have been 45 individuals who have served as mayor in San Francisco since 1850, when California became a state following the American Conquest of California. Prior to the conquest, Californios served as Mayor of San Francisco during the Spanish and Mexican eras since 1779.\n\nThe current mayor is Democrat Daniel Lurie.\n\n## Elections\n\n[edit]\n\nMain articles: 1979 San Francisco mayoral election, 1995, 1999, 2003, 2007, 2011, 2015, 2018, 2019, and 2024 [...] ## Salary and benefits\n\n[edit]\n\nAs of 2024, the mayor is paid an annual salary of $364,582, the highest mayoral salary in the United States. Nine city public employees earned higher salaries than the mayor, including the chief investment officer and the managing dir

In [27]:
agent = create_agent(
    model=model,
    tools=[web_search],
)

question = HumanMessage(content="Who is the current mayor of San Francisco?")

response = agent.invoke(
    {"messages": [question]}
)

In [28]:
from pprint import pprint

pprint(response['messages'])

[HumanMessage(content='Who is the current mayor of San Francisco?', additional_kwargs={}, response_metadata={}, id='70b2c86a-180f-41e9-a0b4-3821e1cd002f'),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 52, 'total_tokens': 73, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_0fff278495', 'id': 'chatcmpl-ETPMJqgzx9jrbLoFc9X7niyikbAPm', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a0ecb5-4c54-7ea3-a98c-0bf7c3c9cb88-0', tool_calls=[{'name': 'web_search', 'args': {'query': 'current mayor of San Francisco 2023'}, 'id': 'call_4RS14Oxl97K1VdPlD0HgMwQQ', 'type': 'tool_call'}], invalid_t

trace: https://smith.langchain.com/public/59432173-0dd6-49e8-9964-b16be6048426/r

In [29]:
response['messages'][-1].content

'The current mayor of San Francisco is Daniel Lurie. He became the 46th mayor of the city and county of San Francisco. If you would like more information about him or his policies, feel free to ask!'

In [30]:
print(response["messages"][1].tool_calls)

[{'name': 'web_search', 'args': {'query': 'current mayor of San Francisco 2023'}, 'id': 'call_4RS14Oxl97K1VdPlD0HgMwQQ', 'type': 'tool_call'}]


In [31]:
tavily_client

TavilyClient(base_url='https://api.tavily.com', requests_made=3)

In [15]:
dir(tavily_client)

['__class__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__firstlineno__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__static_attributes__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_handle_error',
 '_update_usage_stats',
 '_usage_stats',
 'api_key',
 'base_url',
 'batch_search',
 'extract',
 'get_search_suggestions',
 'get_usage_statistics',
 'headers',
 'qna_search',
 'search',
 'timeout',
 'validate_api_key']

In [17]:
tavily_client.validate_api_key()

True